# Travelling Salesman Problem: tutorial answers (slide 22)

Lecture: *Week 9 - Travelling Salesman Problem*, slide 22 (*Tutorial*). Demo: `2 Lecture Demos/1 TSP with Pyomo.ipynb`.

**The problem.** The tutorial has two parts:

1. With the demo notebook: (a) increase the number of cities, when does the MILP become unreasonable to solve?
   (b) change the genetic algorithm's population, crossover and mutation settings. Can it solve larger problems
   than the MILP?
2. Research: why is the TSP hard to solve? What is NP-hard? What is combinatorial optimisation?

**How to run it.** Try the tutorial in the demo notebook first. Then run the first code cell here (it installs Pyomo
and the HiGHS solver) and choose *Run → Run All Cells*. The whole notebook takes about a minute in the browser.

**Expected results.** The cities are the demo's (`np.random.seed(0)`), so the first 20 are the demo's 20 cities and
the first 35 the genetic algorithm's 35. Shortest tours: 5 cities 158.71, 10: 343.89, 15: 381.58, 20: **407.97**,
25: 475.37, 30: 511.32, 35: **518.52**. The genetic algorithm with the demo's settings finds 599.24 on the 35 cities,
**15.6 % longer** than the optimum. Solve times change from computer to computer and from run to run; the lengths
do not.

In [ ]:
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

## The model and the cities (copied from the demo)

The same functions as in the demo notebook, so this notebook runs on its own: the MTZ model, the solver call, the
random cities and their distances. Nothing is plotted here.

In [ ]:
from pyomo.environ import *
import numpy as np
import random
import time

def create_tsp_model(num_cities, distance_matrix):
    model = ConcreteModel()
    model.N = RangeSet(1, num_cities)
    model.x = Var(model.N, model.N, within=Binary, initialize=0)  # x[i,j] = 1: the tour goes from i to j
    model.u = Var(RangeSet(2, num_cities), within=Reals)          # MTZ order variables (city 1 has none)
    model.obj = Objective(expr=sum(distance_matrix[i, j] * model.x[i, j]
                                   for i in model.N for j in model.N if i != j), sense=minimize)
    # one arc leaves and one arc enters every city
    model.out_arc = Constraint(model.N, rule=lambda m, i: sum(m.x[i, j] for j in m.N if j != i) == 1)
    model.in_arc = Constraint(model.N, rule=lambda m, i: sum(m.x[j, i] for j in m.N if j != i) == 1)
    model.no_self_arc = Constraint(model.N, rule=lambda m, i: m.x[i, i] == 0)
    # MTZ: if the tour goes from i to j (both not city 1), then u[j] >= u[i] + 1
    def mtz_rule(m, i, j):
        if i != j and i != 1 and j != 1:
            return m.u[i] - m.u[j] + 1 <= (num_cities - 1) * (1 - m.x[i, j])
        return Constraint.Skip
    model.mtz = Constraint(model.N, model.N, rule=mtz_rule)
    return model

def generate_random_points(num_cities):
    np.random.seed(0)                                # the demo's cities
    return np.random.rand(num_cities, 2) * 100

def compute_distance_matrix(city_coordinates):
    n = len(city_coordinates)
    return {(i + 1, j + 1): np.linalg.norm(city_coordinates[i] - city_coordinates[j])
            for i in range(n) for j in range(n) if i != j}

## 1(a) When does the MILP become unreasonable to solve?

The cell solves the TSP for 5, 10, …, 35 cities and prints the shortest tour and the solve time. It takes about
half a minute in the browser.

In [ ]:
optimum = {}
print(" cities   shortest tour   solve time")
for n in [5, 10, 15, 20, 25, 30, 35]:
    distances = compute_distance_matrix(generate_random_points(n))
    model = create_tsp_model(n, distances)
    start = time.perf_counter()
    SolverFactory('appsi_highs').solve(model)
    seconds = time.perf_counter() - start
    optimum[n] = value(model.obj)
    print(f"{n:6d}   {optimum[n]:13.2f}   {seconds:8.1f} s")

**Answer.** In the browser, up to about 25 cities solve in a second or two, 30 cities take about 6 seconds and 35
cities about 20 seconds; 40 cities took about two and a half minutes (a desktop is several times faster). The
time does not grow smoothly (in the browser 25 cities solved faster than 20) because the solver's work depends on
where the cities lie, not only on how many there are, but the trend is steep: every few extra cities can multiply
the time. For this formulation in the browser, "unreasonable" starts at about 40 cities.

The number of possible tours, (n − 1)!/2, grows far faster: about 10⁴⁶ for 40 cities. The solver does not try
them all. It solves LP relaxations and uses branch and bound to rule out most tours without looking at them, which
is why 35 cities take seconds and not centuries (compare the 22-city claim on slide 6). Specialised solvers such as
Concorde (William Cook and colleagues, slide 11) use much stronger constraints than MTZ and have solved TSPs with
tens of thousands of cities.

## 1(b) The genetic algorithm: settings and quality

The genetic algorithm from the demo, unchanged. It is run on the demo's 35 cities with several settings. Each
run starts from the same seed, so the results repeat, and each length is compared with the optimum 518.52 from
part 1(a).

In [ ]:
def generate_random_population(num_individuals, num_cities):
    return [random.sample(range(1, num_cities + 1), num_cities) for _ in range(num_individuals)]

def compute_fitness(individual, distance_matrix):
    # "fitness" here is the tour length: smaller is better
    fitness = 0
    for i in range(len(individual) - 1):
        fitness += distance_matrix[(individual[i], individual[i + 1])]
    return fitness + distance_matrix[(individual[-1], individual[0])]  # back to the start

def tournament_selection(population, distance_matrix, tournament_size):
    return [min(random.sample(population, tournament_size), key=lambda x: compute_fitness(x, distance_matrix))
            for _ in range(len(population))]

def ordered_crossover(parent1, parent2):
    size = len(parent1)
    offspring1, offspring2 = [-1] * size, [-1] * size
    start, end = sorted(random.sample(range(size), 2))
    offspring1[start:end] = parent1[start:end]
    offspring2[start:end] = parent2[start:end]
    p1_indices, p2_indices = set(parent1[start:end]), set(parent2[start:end])
    p1_remaining = [x for x in parent2 if x not in p1_indices]
    p2_remaining = [x for x in parent1 if x not in p2_indices]
    offspring1[end:] = [x for x in p1_remaining if x not in offspring1]
    offspring1[:start] = [x for x in p1_remaining if x not in offspring1]
    offspring2[end:] = [x for x in p2_remaining if x not in offspring2]
    offspring2[:start] = [x for x in p2_remaining if x not in offspring2]
    return offspring1, offspring2

def swap_mutation(individual):
    idx1, idx2 = random.sample(range(len(individual)), 2)
    individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
    return individual

def genetic_algorithm(num_cities, distance_matrix, population_size, num_generations, tournament_size,
                      crossover_rate, mutation_rate):
    population = generate_random_population(population_size, num_cities)
    for _ in range(num_generations):
        parents = tournament_selection(population, distance_matrix, tournament_size)
        offspring = []
        for i in range(0, len(parents), 2):
            parent1, parent2 = parents[i], parents[i + 1]
            if random.random() < crossover_rate:
                child1, child2 = ordered_crossover(parent1, parent2)
            else:
                child1, child2 = parent1[:], parent2[:]
            if random.random() < mutation_rate:
                child1 = swap_mutation(child1)
            if random.random() < mutation_rate:
                child2 = swap_mutation(child2)
            offspring += [child1, child2]
        # keep the best population_size tours of parents and children together
        population = sorted(population + offspring, key=lambda x: compute_fitness(x, distance_matrix))[:population_size]
    best = min(population, key=lambda x: compute_fitness(x, distance_matrix))
    return best, compute_fitness(best, distance_matrix)

In [ ]:
distances35 = compute_distance_matrix(generate_random_points(35))
settings = [  # (population, generations, crossover rate, mutation rate)
    (100, 200, 0.7, 0.2),    # the demo's settings
    (100, 200, 0.0, 0.2),    # no crossover
    (100, 200, 0.7, 0.8),    # more mutation
    (300, 200, 0.7, 0.2),    # three times the population
    (100, 1000, 0.7, 0.2),   # five times the generations
]
print("population  generations  crossover  mutation   length   above optimum   time")
for pop, gens, cx, mut in settings:
    random.seed(1)                                   # the demo's seed
    start = time.perf_counter()
    tour, length = genetic_algorithm(35, distances35, pop, gens, 5, cx, mut)
    seconds = time.perf_counter() - start
    gap = 100 * (length - optimum[35]) / optimum[35]
    print(f"{pop:10d}  {gens:11d}  {cx:9.1f}  {mut:8.1f}  {length:7.2f}  {gap:12.1f} %  {seconds:5.1f} s")

**Answer.** On these 35 cities the demo's settings stop 15.6 % above the shortest tour. Without crossover the
result is much worse (35.7 %): crossover is what combines good parts of different tours. More mutation (0.8) gives
the best result here (10.5 %), and three times the population also helps (11.4 %) but takes three times as long.
Five times the generations hardly helps (14.4 %): after a while the population consists of nearly identical tours,
and only mutation still changes them. Each line is a single run with one seed; with another seed the numbers
change, so to compare settings fairly run each several times.

**Can it solve larger problems than the MILP?** It always returns a tour, quickly, for any number of cities: its
time grows roughly with population × generations × cities. But it gives no guarantee, not even a bound on how far
from the shortest tour it is. On 35 cities the MILP found the shortest tour, and proved it is the shortest, in
about 20 seconds in the browser, while the best run here is still 10 % longer. For hundreds or thousands of cities
an exact MILP like this one is out of reach and heuristics are the practical choice; good ones for the TSP
(2-opt, Lin–Kernighan) usually come within a few per cent of the optimum.

## 2. Research questions

**Why is the TSP hard to solve?** The number of tours, (n − 1)!/2, grows faster than any power of n (slides 3–6),
and no known algorithm avoids an exponential worst case: the best exact method known for general instances, the
Held–Karp dynamic program (1962), takes about n²·2ⁿ steps. Branch and bound (as HiGHS does above) solves many
instances quickly, but its worst case still grows exponentially.

**What is NP-hard?** NP ("nondeterministic polynomial time", not "non-polynomial") is the set of yes/no problems
whose *yes* answers can be **checked** in polynomial time: given a tour, it is easy to check that its length is at
most L. A problem is **NP-hard** if every problem in NP can be translated into it in polynomial time, so it is at
least as hard as any problem in NP. **NP-complete** means NP-hard *and* in NP. The yes/no TSP ("is there a tour of
length at most L?") is NP-complete; finding the shortest tour is NP-hard. No polynomial-time algorithm is known for
any NP-hard problem; finding one, or proving that none exists, is the P vs NP question, one of the Millennium
Prize Problems (slide 8).

**What is combinatorial optimisation?** Choosing the best option from a finite but usually enormous set of
discrete options: orders, subsets, assignments, routes. Examples from this module: the TSP (an order of cities),
the knapsack and set covering problems (Week 05: subsets), the assignment problem (Week 06), job shop scheduling
(Week 07: orders of jobs), shortest paths and minimum spanning trees (Week 08). Integer programming is one general
way to solve them. Some (shortest paths, spanning trees, assignment) have fast special algorithms; others (TSP,
knapsack, job shop) are NP-hard.